# 15.5 Test Doubles — Mocking, Patching and Faking

**Prerequisites:** 15.4 Fixtures, 05 OOPs, 07 Module and Packages, 11.5 HTTP  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The vocabulary: dummy, stub, spy, mock, fake — and why the difference matters
- `unittest.mock.Mock` — `return_value`, `side_effect`, and the call assertions
- 🔴 **Where to patch** — patch where the name is *looked up*, not where it is defined
- 🔴 `autospec` catching a wrong call that a plain `Mock` accepts in silence
- `monkeypatch` as the pytest-native alternative
- Faking the clock and an HTTP client — and why **injection beats patching**
- 🔴 When mocking is the wrong answer: green tests over broken production

---

## The problem

A unit test is supposed to be **fast, isolated and repeatable** (**15.1**). Real dependencies
are none of those:

| Dependency | Why it ruins the test |
|---|---|
| An HTTP API | slow, offline in CI, rate-limited, and its data changes |
| The system clock | `test_expiry` passes in the morning and fails at 23:59 |
| A database | slow, needs a server, leaks state between tests |
| `random` | different answer every run |
| Sending email / charging a card | 🔴 you really do not want the test doing this |

A **test double** is a stand-in you control. The name is from film: a stunt double stands in
for the actor for the dangerous parts.

> `pytest` has no mocking of its own. The standard library's **`unittest.mock`** is the tool,
> whichever runner you use — which is why **15.2** matters even if you never write a `TestCase`.

## The five kinds, and the two that matter

The vocabulary is Gerard Meszaros's, and it gets used loosely. The distinction worth keeping is
the last column.

| Kind | What it is | Does the test assert on it? |
|---|---|---|
| **Dummy** | a value passed only to fill a parameter | no |
| **Stub** | returns canned answers | no — you assert on the *result* |
| **Spy** | a real-ish object that records how it was called | sometimes |
| **Mock** | pre-programmed with expectations about calls | **yes — the calls are the assertion** |
| **Fake** | a working, simplified implementation (in-memory DB) | no |

🔴 **Stub vs mock is a real distinction, not pedantry.**

- A **stub** lets you set up a situation: *"pretend the API returns 500."* You then assert on
  what your code **did**.
- A **mock** asserts on the **interaction**: *"the retry handler must call `sleep` exactly
  three times."* You are testing *how* your code works, not *what* it produces.

Mocks are the ones that rot. Every mock assertion is a claim about your own implementation, so
it breaks when you refactor — even when the behaviour is unchanged. Prefer stubs and fakes;
use mocks when the interaction genuinely *is* the behaviour (an email must be sent exactly
once; the card must be charged exactly once).

In [ ]:
from unittest.mock import Mock, MagicMock, call

# A Mock invents attributes and methods on demand, and records everything.
store = Mock()

store.save("cache:user:7", {"name": "Ada"}, ttl=300)
store.save("cache:user:9", {"name": "Alan"}, ttl=60)
store.delete("cache:user:7")

print("called          :", store.save.called)
print("call count      :", store.save.call_count)
print("last call       :", store.save.call_args)
print("all save calls  :", store.save.call_args_list)
print("every call, any :", store.mock_calls)

# The call assertions - these raise AssertionError when they do not hold.
store.save.assert_called()
store.save.assert_any_call("cache:user:9", {"name": "Alan"}, ttl=60)
store.delete.assert_called_once_with("cache:user:7")
store.save.assert_has_calls([
    call("cache:user:7", {"name": "Ada"}, ttl=300),
    call("cache:user:9", {"name": "Alan"}, ttl=60),
])
print("\nall call assertions held")

### `return_value` and `side_effect`

| Attribute | Effect |
|---|---|
| `mock.return_value = x` | every call returns `x` |
| `mock.side_effect = [a, b, c]` | successive calls return `a`, then `b`, then `c`, then `StopIteration` |
| `mock.side_effect = SomeError` | every call **raises** |
| `mock.side_effect = fn` | every call delegates to `fn` |

`side_effect` with a list is how you test retry logic: *"fail, fail, then succeed."*

In [ ]:
import time


def fetch_with_retry(client, url, attempts=3, sleep=time.sleep):
    """Fetch `url`, retrying on ConnectionError with a fixed backoff."""
    last_error = None
    for attempt in range(attempts):
        try:
            return client.get(url)
        except ConnectionError as exc:
            last_error = exc
            if attempt < attempts - 1:
                sleep(2 ** attempt)
    raise last_error


# --- a STUB: canned answers, set up so we can assert on the RESULT ---
client = Mock()
client.get.side_effect = [
    ConnectionError("connection reset"),
    ConnectionError("connection reset"),
    {"status": 200, "body": "ok"},
]
slept = Mock()                     # a SPY: records the sleeps, so the test is instant

result = fetch_with_retry(client, "https://api.example.com/jobs", sleep=slept)

print("result        :", result)
print("get attempts  :", client.get.call_count)
print("sleeps        :", [c.args[0] for c in slept.call_args_list])

# --- side_effect as an exception class: every call fails ---
always_failing = Mock()
always_failing.get.side_effect = ConnectionError("host unreachable")
try:
    fetch_with_retry(always_failing, "https://api.example.com/jobs", sleep=Mock())
except ConnectionError as exc:
    print("\ngave up after", always_failing.get.call_count, "attempts:", exc)

Three `get` calls, two sleeps of **1** and **2** seconds — and the test took
no time at all, because `slept` only recorded the numbers.

🔴 Notice what made that easy: `fetch_with_retry` takes `client` and `sleep` **as parameters**.
No patching was needed at all. Keep that in mind — it is the theme of the last section.

## `Mock` vs `MagicMock`

`Mock` does not support the dunder protocols; `MagicMock` configures them all. If you need
`len(obj)`, `obj[key]`, `iter(obj)`, `obj + x` or `with obj:` — you need `MagicMock`.

In [ ]:
plain = Mock()
magic = MagicMock()

for label, double in (("Mock", plain), ("MagicMock", magic)):
    for op_name, operation in (
        ("len(x)",      len),
        ("x['key']",    lambda x: x["key"]),
        ("list(x)",     list),
        ("bool(x)",     bool),
        ("with x:",     lambda x: x.__enter__()),
    ):
        try:
            operation(double)
            outcome = "works"
        except (TypeError, AttributeError) as exc:
            # 🔴 note the two different exception types: unsupported protocols
            # raise TypeError, but a missing dunder attribute raises AttributeError,
            # because Mock auto-creates ordinary attributes and dunders alike are not.
            outcome = f"{type(exc).__name__}: {str(exc)[:40]}"
        print(f"  {label:<10} {op_name:<10} {outcome}")
    print()

print("MagicMock's defaults are configured, not invented:")
print("  len(MagicMock())  =", len(MagicMock()))
print("  bool(MagicMock()) =", bool(MagicMock()))
print("  list(MagicMock()) =", list(MagicMock()))

## 🔴 Where to patch

This is the single biggest source of "my mock isn't working", and the rule is short:

> **Patch the name where it is *looked up*, not where it is *defined*.**

Why: `from clock import now` **binds a new name** in the importing module (**07**). Two names,
one function object. Replacing `clock.now` does nothing to `scheduler.now`, because
`scheduler` already has its own reference.

```
   clock.py                    scheduler.py
   ┌──────────────┐            ┌────────────────────────────┐
   │ def now(): …─┼────────────┼─> now      (its own name!) │
   └──────────────┘            │   def next_run(interval):  │
         ▲                     │       return now() + …     │
         │                     └────────────────────────────┘
   patch("clock.now")                    ▲
   rebinds THIS one                      │
   — scheduler never looks here    patch("scheduler.now")
                                   rebinds the one actually used ✅
```

The next cell runs both, in a real project, so you can see one fail and one pass.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py155_"))


def make_project(files, name="proj"):
    project = Path(tempfile.mkdtemp(prefix=f"{name}_", dir=WORK))
    for relpath, source in files.items():
        path = project / relpath
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return project


def pytest_in(project, *args):
    cmd = [sys.executable, "-m", "pytest", "--no-header", "-p", "no:cacheprovider", *args]
    done = subprocess.run(cmd, cwd=project, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=300)
    return (f"$ pytest {' '.join(args)}".rstrip() + "\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip()
            + "\n" + "-" * 70 + f"\nexit code: {done.returncode}")


where = make_project({
    "clock.py": r"""
        import time


        def now():
            return time.time()
    """,
    "scheduler.py": r"""
        from clock import now          # <- binds `now` INTO this module


        def next_run(interval):
            return now() + interval
    """,
    "test_where.py": r"""
        from unittest.mock import patch

        import scheduler


        def test_patching_where_it_is_DEFINED_does_not_work():
            with patch("clock.now", return_value=1000.0):
                assert scheduler.next_run(60) == 1060.0


        def test_patching_where_it_is_LOOKED_UP_works():
            with patch("scheduler.now", return_value=1000.0):
                assert scheduler.next_run(60) == 1060.0
    """,
}, name="where")

print(pytest_in(where, "-q", "--tb=short"))

The failing test got a real Unix timestamp back — the patch had no effect
whatsoever. And note that it failed **loudly**, which is lucky. The genuinely dangerous version
is a patch that silently does nothing while your test still passes for some other reason.

### The corollary

If `scheduler.py` had used `import clock` and called `clock.now()`, then `patch("clock.now")`
**would** work — because the lookup happens at call time, through the module object. That is
one small argument for `import module` over `from module import name`.

| Import style in the code under test | Patch target |
|---|---|
| `from clock import now` → `now()` | `"scheduler.now"` |
| `import clock` → `clock.now()` | `"clock.now"` |

## 🔴 `autospec` — the mock that checks the signature

A plain `Mock` accepts **anything**. Wrong arguments, wrong number of them, methods that do not
exist — all silently fine. That means a test can keep passing after you rename a method or
change its signature, while production breaks.

`create_autospec` (and `patch(..., autospec=True)`) builds the double **from the real object**,
so it enforces the real API.

In [ ]:
autospec_demo = make_project({
    "store.py": r"""
        class CacheStore:
            def save(self, key, value, ttl=None):
                raise RuntimeError("the real store was touched!")

            def delete(self, key):
                raise RuntimeError("the real store was touched!")
    """,
    "test_autospec.py": r"""
        from unittest.mock import Mock, create_autospec

        import store


        def test_a_plain_mock_accepts_nonsense():
            fake = Mock()
            fake.save("k", "v", "extra", "positional", nonsense=1)   # 🔴 accepted
            fake.saev("typo in the method name")                     # 🔴 also accepted
            fake.completely_invented_method()                        # 🔴 and this
            assert fake.saev.called          # the typo is now "verified"!


        def test_autospec_rejects_too_many_arguments():
            fake = create_autospec(store.CacheStore, instance=True)
            fake.save("k", "v")                                      # fine
            try:
                fake.save("k", "v", "extra", "positional")
            except TypeError as exc:
                print("      autospec caught:", exc)
            else:
                raise AssertionError("autospec should have rejected that call")


        def test_autospec_rejects_a_misspelled_method():
            fake = create_autospec(store.CacheStore, instance=True)
            try:
                fake.saev("typo")
            except AttributeError as exc:
                print("      autospec caught:", exc)
            else:
                raise AssertionError("autospec should have rejected the typo")
    """,
}, name="autospec")

print(pytest_in(autospec_demo, "-q", "-s"))

Read `test_a_plain_mock_accepts_nonsense` again: it **passes**, and its final
assertion cheerfully verifies that a **misspelled method** was called. A suite full of plain
`Mock`s can be entirely green against an API that no longer exists.

🔴 **Use `autospec=True` by default.** The cost is a slightly slower mock; the benefit is that
your doubles break when the real thing changes — which is the entire point of having tests.

## `monkeypatch` — the pytest-native alternative

`monkeypatch` (**15.4**) does the same job as `patch`, with fixture-managed undo and a lighter
syntax. It is usually the better choice inside pytest.

| `unittest.mock` | `monkeypatch` |
|---|---|
| `with patch("mod.name", new)` | `monkeypatch.setattr("mod.name", new)` |
| `patch.dict(os.environ, {...})` | `monkeypatch.setenv("KEY", "value")` |
| `patch.object(obj, "attr", new)` | `monkeypatch.setattr(obj, "attr", new)` |
| — | `monkeypatch.delenv`, `monkeypatch.chdir`, `monkeypatch.syspath_prepend` |

The same "where to patch" rule applies to both — it is a property of Python's import system,
not of the tool.

In [ ]:
monkey = make_project({
    "clock.py": r"""
        import time


        def now():
            return time.time()
    """,
    "billing.py": r"""
        from clock import now

        GRACE_SECONDS = 7 * 24 * 3600


        def is_expired(subscription_ends_at):
            return now() > subscription_ends_at + GRACE_SECONDS
    """,
    "test_monkey.py": r"""
        import billing


        def test_not_expired_during_the_grace_period(monkeypatch):
            ends_at = 1_000_000.0
            monkeypatch.setattr("billing.now", lambda: ends_at + 3 * 24 * 3600)
            assert billing.is_expired(ends_at) is False


        def test_expired_after_the_grace_period(monkeypatch):
            ends_at = 1_000_000.0
            monkeypatch.setattr("billing.now", lambda: ends_at + 8 * 24 * 3600)
            assert billing.is_expired(ends_at) is True


        def test_exactly_on_the_boundary(monkeypatch):
            ends_at = 1_000_000.0
            monkeypatch.setattr("billing.now", lambda: ends_at + 7 * 24 * 3600)
            assert billing.is_expired(ends_at) is False       # `>` not `>=`


        def test_the_patch_is_undone_afterwards():
            import clock
            assert billing.now is clock.now
    """,
}, name="monkey")

print(pytest_in(monkey, "-v"))

Three tests pin down the boundary of a rule that would otherwise be
**untestable** — you cannot wait eight days — and the fourth proves `monkeypatch` put the real
function back.

## 🔴 Injection beats patching

Everything above patches a global. It works, but every patch is a claim about *module
structure*: "`billing` looks up a name called `now`". Rename the import and the test breaks
even though the behaviour did not.

**Passing the dependency in** removes the problem entirely. Compare:

```python
# patched:  the test must know how billing imports its clock
def is_expired(ends_at):
    return now() > ends_at + GRACE

# injected: the test just passes a different clock
def is_expired(ends_at, now=time.time):
    return now() > ends_at + GRACE
```

The injected version needs **no mocking library at all** — you saw this already with
`fetch_with_retry(..., sleep=slept)`. A plain function, or a small **fake** class, is enough.

> **The rule of thumb:** if a test is hard to write without patching, that is usually the test
> telling you about a design problem, not about a missing tool.

In [ ]:
from dataclasses import dataclass, field


class FakeCacheStore:
    """A working in-memory stand-in - a FAKE, not a mock.

    It implements the real behaviour simply enough to test against, so tests
    read as 'given a cache containing X' rather than 'given save() returns Y'.
    """

    def __init__(self):
        self.data = {}

    def save(self, key, value, ttl=None):
        self.data[key] = (value, ttl)

    def get(self, key):
        entry = self.data.get(key)
        return entry[0] if entry else None

    def delete(self, key):
        self.data.pop(key, None)


@dataclass
class ProfileService:
    """Loads user profiles, caching them. Both dependencies are injected."""
    store: object
    loader: object
    ttl: int = 300
    loads: int = field(default=0, init=False)

    def profile(self, user_id):
        key = f"cache:user:{user_id}"
        cached = self.store.get(key)
        if cached is not None:
            return cached
        self.loads += 1
        fresh = self.loader(user_id)
        self.store.save(key, fresh, ttl=self.ttl)
        return fresh


cache = FakeCacheStore()
service = ProfileService(store=cache, loader=lambda uid: {"id": uid, "name": "Ada"})

first = service.profile(7)
second = service.profile(7)
third = service.profile(9)

print("first == second      :", first == second)
print("backend loads        :", service.loads, "(not 3 - the second call was cached)")
print("cache contents       :", sorted(cache.data))
print("ttl actually stored  :", cache.data["cache:user:7"][1])

# The fake is a real object, so you can set up situations directly:
cache.delete("cache:user:7")
service.profile(7)
print("\nafter eviction, loads:", service.loads)
print("\nNo patching, no Mock, no import tricks - just a smaller implementation.")

## 🔴 When mocking is the wrong answer

Three rules that will save you more pain than any amount of `mock` API knowledge.

### 1. Don't mock what you don't own

Mocking `requests.get` encodes **your belief** about how `requests` behaves. If that belief is
wrong — or the library changes — your tests stay green and production breaks. They are testing
your assumptions, not the integration.

Instead: wrap the third-party library in a thin adapter **you** own, mock the adapter in unit
tests, and write a small number of **integration tests** against the real thing (**15.1**'s
pyramid, and **11.5**).

### 2. A fake that drifts from the real thing is worse than no test

The `autospec` demo above is this failure in miniature. A hand-written fake has the same
problem: nothing forces it to stay in step. Defend it by running the **same test suite** against
both the fake and the real implementation — a parametrised fixture (**15.4**) with
`params=["fake", "real"]` does exactly this.

### 3. Asserting on calls tests the implementation

`store.save.assert_called_once_with(...)` says *"the code must call save exactly once with
these arguments"*. Refactor `save` into `save_many` and the test fails, though nothing a user
can observe has changed.

Ask: **would this test have to change if I rewrote the internals but kept the behaviour?** If
yes, you are testing the mechanism. Sometimes that is right — an email must be sent exactly
once — but it should be a decision, not an accident.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Patching where the function is defined instead of where it is looked up.** `from x import y` binds a second name; `patch("x.y")` does not touch it.
2. 🔴 **Plain `Mock` instead of `autospec=True`.** It accepts misspelled methods and wrong signatures, so tests stay green against an API that no longer exists.
3. **Mocking what you do not own** — `requests`, the database driver, the cloud SDK. You end up testing your beliefs about someone else's library.
4. **Asserting on calls when you could assert on the result.** Every call assertion is a claim about implementation and breaks on refactors.
5. **Forgetting `Mock` has no dunders.** `len(mock)` and `mock[key]` need `MagicMock`.
6. **A `Mock()` attribute used as a boolean.** Every attribute is a truthy auto-created Mock, so `if mock.is_enabled:` is always true — including when the real attribute would be `False`.
7. **Leaving a patch active.** Use `with patch(...)`, the decorator form, or `monkeypatch` — never a bare `patch(...).start()` without a matching `stop()`.
8. **A hand-written fake that drifts.** Run the same tests against the fake and the real implementation, or it will quietly stop meaning anything.
9. **Reaching for a mock before considering a parameter.** If the dependency can be passed in, no mocking library is needed at all.

## Best Practices

- Prefer **injection** to patching: a default argument or a constructor parameter beats any amount of `patch`.
- Prefer a **fake** to a mock when the dependency has real behaviour worth simulating.
- Always pass `autospec=True` (or use `create_autospec`) so the double follows the real API.
- Inside pytest, reach for `monkeypatch` first — it undoes itself.
- Assert on **outcomes**; assert on calls only when the call *is* the observable behaviour (an email sent, a card charged).
- Wrap third-party libraries in an adapter you own, and mock the adapter.
- Keep a handful of integration tests against the real dependency to catch fake drift.
- Name doubles for what they are: `fake_store`, `stub_loader`, `spy_sleep`.

## Practice Exercises

Try these before moving on.

1. Rewrite `fetch_with_retry`'s test using `patch` instead of parameter injection. Which version is shorter, and which would survive renaming the import?
2. 🔴 Reproduce the where-to-patch failure yourself, then change the module under test to `import clock` / `clock.now()` and confirm `patch("clock.now")` now works.
3. Take `test_a_plain_mock_accepts_nonsense` and add `autospec`. List every line that now fails, and say what real bug each one would have caught.
4. Write a `FakeCacheStore` that expires entries by TTL, using an injected clock. Test that an entry is a hit before expiry and a miss after — without sleeping.
5. Use `side_effect` with a list to simulate an API that fails twice then succeeds, and assert the caller retried exactly three times and slept 1s then 2s.
6. 🔴 Write a test that passes while the code is broken, by mocking too much. Then fix the test so it would have caught the bug.
7. Parametrise a fixture with `params=["fake", "real"]` so one test suite runs against both your `FakeCacheStore` and a real dict-backed store. What did it catch?
8. **Interview question:** what is the difference between a stub and a mock, and why does a suite full of mocks get harder to refactor over time?

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | `unittest.mock` gained `ThreadingMock`, for asserting a call happened on another thread (**12.2**) |
| **3.11** | `Mock` no longer auto-creates attributes matching `assret_*`/`assrt_*` typos — a long-standing footgun where `mock.assret_called()` silently passed |
| **3.10** | `patch` supports `create_autospec` improvements for `@dataclass` targets (**5.3**) |
| **3.8** | `AsyncMock` added — the double for `async def` (**12.5**); `Mock` does not work on coroutines |
| **3.8** | `Mock.call_args` gained `.args` and `.kwargs`, used in this notebook |

🔴 If you test `async` code, you need **`AsyncMock`**, not `Mock` — a plain `Mock` returns a
`Mock` where the caller awaits a coroutine, and fails with `TypeError: object Mock can't be
used in 'await' expression`. See **12.5**.

## Where next

| Notebook | Covers |
|---|---|
| **15.6 Testing in Practice** | coverage, property-based testing, layout, CI — the last piece |

## Related

- **15.4 Fixtures** — `monkeypatch`, and parametrised fixtures for fake-vs-real
- **15.2 unittest** — where `unittest.mock` lives
- **07 Module and Packages** — the import binding that makes "where to patch" a question at all
- **05 OOPs** / **5.4 Duck Typing and Protocols** — why a fake only needs the right shape
- **11.5 HTTP** and **19 Working with APIs** — the dependency most worth faking
- **12.5 asyncio** — `AsyncMock`